(course-core-selection-basics)=

# Module 7: Selection Mechanism

**Welcome back, Apprentice Master.** In [Module 6: Visualizing Anything](../Common_Core/06_Visualizing_Anything.ipynb), you gained the eyes to inspect molecular systems in 3D. Now, you need the precision targeting system: **MolSysMT's Selection Mechanism**.

In large molecular systems containing thousands of atoms, water molecules, and co-crystallized ligands, you rarely perform calculations or extractions on the whole system at once. Instead, you target specific subsets—such as alpha carbons, active site residues, bound inhibitors, or binding-pocket water molecules.

In MolSysMT, the universal function **`msm.select()`** acts as your query engine. It accepts direct index lists, boolean attribute expressions, element-scoped queries, spatial proximity criteria, and topological connectivity operators.

```{admonition} Glossary: Selection Mechanism
:class: dropdown info
The **Selection Mechanism** is MolSysMT's unified querying engine (`msm.select()`) for targeting specific subsets of a molecular system. It evaluates index arrays, boolean attribute expressions, element-scoped queries, spatial distance criteria, and topological connectivity across any data form. Third-party syntax support is directional: `syntax=...` parses an input query, while `to_syntax=...` translates selected indices. See the {ref}`user-foundations-support-selection-syntaxes` guide for the executable capability matrix.
```

(course-core-selection-basics-learning-outcomes)=
```{admonition} Learning Outcomes
:class: dropdown learning-outcomes

By the end of this module, you will be able to:
- Perform direct index and list-based selections.
- Build boolean attribute selection expressions (`atom_name`, `group_name`, `molecule_type`).
- Combine query conditions using logical operators (`and`, `or`, `not`).
- Scope selection outputs to specific hierarchical elements (`element='group'`, `element='molecule'`, `element='chain'`).
- Perform spatial distance selections (`within X nanometers of ...`).
- Target atoms by topological bond connectivity (`all bonded to ...`).
```

### 1. Index & List-Based Selections

Let's begin by importing MolSysMT and loading our T4 Lysozyme demonstration system.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load T4 Lysozyme file
lysozyme = systems['T4 lysozyme L99A']['181l.bcif.gz']

The simplest selection is selecting all items using `'all'`, or passing explicit integer indices or lists of indices:

In [2]:
# Select all atom indices
all_atoms = msm.select(lysozyme, selection='all')
print(f"Total selected atoms: {len(all_atoms)}")

# Select specific atom indices by passing a Python list
first_three = msm.select(lysozyme, selection=[0, 1, 2])
print(f"Selected explicit indices: {first_three}")

Total selected atoms: 1441
Selected explicit indices: [0, 1, 2]


### 2. Boolean Attribute Expressions & Logical Operators

You can build expressive query strings using topological, structural, and physical attributes (`atom_name`, `atom_type`, `group_name`, `group_id`, `molecule_type`, `chain_id`).

Logical operators (`and`, `or`, `not`) allow you to combine multiple criteria:

In [3]:
# Select all alpha-carbons (CA atoms)
ca_atoms = msm.select(lysozyme, selection='atom_name == "CA"')
print(f"Total CA atoms: {len(ca_atoms)}")

# Combine conditions: CA atoms belonging to Alanine groups
ala_ca_atoms = msm.select(lysozyme, selection='group_name == "ALA" and atom_name == "CA"')
print(f"Total Alanine CA atoms: {len(ala_ca_atoms)}")

Total CA atoms: 162
Total Alanine CA atoms: 17


:::{hint}
:class: dropdown
**msm.select()**: Evaluates selection expressions and returns indices matching the query. See API doc: {func}`molsysmt.basic.select`.
:::

### 3. Element-Scoped Selections

By default, **`msm.select()`** returns **atom indices**. As you learned in [Module 3: Molecular Elements](../Common_Core/03_Molecular_Elements.ipynb), you can use the **`element`** parameter to return indices for specific structural levels (`group`, `molecule`, `chain`, `entity`):

In [4]:
# Select Alanine groups (returns group indices)
ala_group_indices = msm.select(lysozyme, element='group', selection='group_name == "ALA"')
print(f"Alanine group indices: {ala_group_indices}")

# Select water molecules (returns molecule indices)
water_mol_indices = msm.select(lysozyme, element='molecule', selection='molecule_type == "water"')
print(f"Total water molecule indices: {len(water_mol_indices)}")

Alanine group indices: [40, 41, 48, 62, 72, 73, 81, 92, 96, 97, 98, 111, 128, 129, 133, 145, 159]


Total water molecule indices: 136


### 4. Spatial & Proximity Selections

MolSysMT supports distance-based proximity selections using the **`within`** operator. This allows you to identify atoms or groups within a physical distance threshold of a reference selection:

In [5]:
# Select all atoms within 0.5 nanometers of atom index 0
nearby_atoms = msm.select(lysozyme, selection='all within 0.5 nanometers of atom_index == 0')
print(f"Atoms within 0.5 nm of atom 0: {nearby_atoms}")

Atoms within 0.5 nm of atom 0: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 41, 42, 1247, 1249, 1319, 1353]


### 5. Connectivity Selections

Beyond spatial distance, MolSysMT supports topological connectivity operators such as **`all bonded to`**. This enables selecting all atoms directly bound to a target atom or group:

A valid empty operand follows set semantics: `all bonded to atom_type=="Xe"` returns an empty selection when there are no xenon atoms; `all not bonded to atom_type=="Xe"` retains the left selection. Empty matches do not make the expression malformed. Missing chemical information still raises its documented diagnostic.


In [6]:
# Select all atoms covalently bonded to atom index 10
bonded_atoms = msm.select(lysozyme, selection='all bonded to atom_index == 10')
print(f"Atoms bonded to atom 10: {bonded_atoms}")

Atoms bonded to atom 10: [9, 11, 16]


**Restricting queries to a scope**

A selection describes what you want to inspect; `mask` limits where the query
may look. `msm.select`, `msm.get` and `msm.info` use masks at the requested element
level. Atom masks do not automatically become group masks. A system-level info
summary uses an atom mask to count the retained inventory. Active masks use
sorted set intersections, so repetition cannot add members to the result.


In [3]:
import molsysmt as msm
molsys = msm.convert(msm.systems['T4 lysozyme L99A']['181l.pdb'],
                     to_form='molsysmt.MolSys')
scope = [0, 1, 2, 3]
selected = msm.select(molsys, selection=[5, 3, 1], mask=scope)
summary = msm.info(molsys, element='atom', selection=[5, 3, 1],
                   mask=scope, output_type='dataframe')
assert selected == [1, 3]
assert summary['index'].tolist() == selected
print(selected)


[1, 3]


--- 

### 🏆 Challenge 7: The Selection Master

1. Load the **SARS-CoV-2 Protease** using its PDB ID: `'pdb_id:6LU7'`.
2. Select all backbone nitrogen atoms (`atom_name == "N"`) in the protein.
3. Select all group indices (`element='group'`) that correspond to Histidine (`group_name == "HIS"`).
4. Select all atoms within 0.4 nanometers of the bound inhibitor ligand (`molecule_type == "small molecule"`).

Mastering selection expressions gives you surgical control over any molecular system. In [Module 8: Extracting Molecular Attributes](../Common_Core/08_Extracting_Molecular_Attributes.ipynb), we will combine selections with `msm.get()` to extract custom datasets programmatically.

```{key-takeaway}
The `msm.select()` function is MolSysMT's universal query engine. Combining boolean attribute expressions, element scoping (`element=...`), spatial distance operators (`within`), and topological connectivity (`all bonded to`) provides surgical control over complex molecular systems.
```

(course-core-selection-basics-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.basic.select` — Unified selection and querying engine.

**Related Course Modules & Guides:**
- Previous Module: [Module 6: Visualizing Anything](../Common_Core/06_Visualizing_Anything.ipynb)
- Next Module: [Module 8: Extracting Molecular Attributes](../Common_Core/08_Extracting_Molecular_Attributes.ipynb)
- User Guide: {ref}`user-foundations-index`
:::